In [1]:
!pip install kagglehub pandas scikit-learn

Defaulting to user installation because normal site-packages is not writeable


In [2]:
import kagglehub
import pandas as pd
import os

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

In [3]:
path = kagglehub.dataset_download("jrobischon/wikipedia-movie-plots")

print("Dataset downloaded at:", path)

Dataset downloaded at: C:\Users\harikesh s\.cache\kagglehub\datasets\jrobischon\wikipedia-movie-plots\versions\1


In [4]:
file_path = os.path.join(path, "wiki_movie_plots_deduped.csv")

data = pd.read_csv(file_path)

print("Dataset Shape:", data.shape)

data.head()

Dataset Shape: (34886, 8)


,Release Year,Title,Origin/Ethnicity,Director,Cast,Genre,Wiki Page,Plot
0,1901,Kansas Saloon Smashers,American,Unknown,NaN,unknown,https://en.wikipedia.org/wiki/Kansas_Saloon_Sm...,"A bartender is working at a saloon, serving dr..."
1,1901,Love by the Light of the Moon,American,Unknown,NaN,unknown,https://en.wikipedia.org/wiki/Love_by_the_Ligh...,"The moon, painted with a smiling face hangs ov..."
2,1901,The Martyred Presidents,American,Unknown,NaN,unknown,https://en.wikipedia.org/wiki/The_Martyred_Pre...,"The film, just over a minute long, is composed..."
3,1901,"Terrible Teddy, the Grizzly King",American,Unknown,NaN,unknown,"https://en.wikipedia.org/wiki/Terrible_Teddy,_...",Lasting just 61 seconds and consisting of two ...
4,1902,Jack and the Beanstalk,American,"George S. Fleming, Edwin S. Porter",NaN,unknown,https://en.wikipedia.org/wiki/Jack_and_the_Bea...,The earliest known adaptation of the classic f...


In [5]:
data = data[['Plot','Genre']]
data = data.dropna()
data = data.sample(10000, random_state=42)
data.head()

,Plot,Genre
5337,"When a flying saucer lands in Washington, D.C....",science fiction
9809,"One night at Camp Blackfoot, several campers p...",horror
24075,"The first Asian Nobel Laureate, Rabindranath T...",suspense / drama
19057,A major international financier is found dead ...,detective
24991,Inspector Amar and Inspector Chhaya are after ...,unknown


In [17]:
# clean genre column
data['Genre'] = data['Genre'].str.lower()
# remove bad values
data = data[~data['Genre'].isin(['unknown','', 'nan'])]
# keep only first genre
data['Genre'] = data['Genre'].apply(lambda x: x.split('|')[0])
# keep only top 8 genres (reduces imbalance)
top_genres = data['Genre'].value_counts().head(8).index
data = data[data['Genre'].isin(top_genres)]
print(data['Genre'].value_counts())

Genre
drama       1731
comedy      1213
action       310
horror       307
thriller     289
romance      287
western      248
crime        169
Name: count, dtype: int64


In [22]:
min_count = data['Genre'].value_counts().min()

balanced_data = (
    data.groupby('Genre', group_keys=False)
    .sample(n=min_count, random_state=42)
)

data = balanced_data.reset_index(drop=True)

print(data['Genre'].value_counts())

Genre
action      169
comedy      169
crime       169
drama       169
horror      169
romance     169
thriller    169
western     169
Name: count, dtype: int64


In [35]:
vectorizer = TfidfVectorizer(
    stop_words='english',
    max_features=8000,
    ngram_range=(1,2)
)

X = vectorizer.fit_transform(data['Plot'])
y = data['Genre']

In [36]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [38]:
model = MultinomialNB()
model.fit(X_train, y_train)
print("Model trained successfully")

Model trained successfully


In [33]:
predictions = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, predictions))

print(classification_report(y_test, predictions, zero_division=0))

Accuracy: 0.47601476014760147
              precision    recall  f1-score   support

      action       0.48      0.34      0.40        29
      comedy       0.29      0.26      0.27        35
       crime       0.47      0.58      0.52        36
       drama       0.33      0.09      0.15        43
      horror       0.53      0.90      0.67        30
     romance       0.46      0.81      0.59        31
    thriller       0.33      0.14      0.20        36
     western       0.67      0.90      0.77        31

    accuracy                           0.48       271
   macro avg       0.44      0.50      0.44       271
weighted avg       0.44      0.48      0.42       271



In [39]:
sample_plot = ["A detective investigates a brutal murder in a small town"]

sample_vector = vectorizer.transform(sample_plot)

print("Predicted Genre:", model.predict(sample_vector)[0])

Predicted Genre: crime


In [40]:
import pickle

pickle.dump(model, open("movie_genre_model.pkl","wb"))
pickle.dump(vectorizer, open("tfidf_vectorizer.pkl","wb"))

print("Model saved successfully")

Model saved successfully
